## Dependencies

In [18]:
## libraries
import os
import sys
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

## project root
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

## modules
from src.evaluators.config import FEAT_X, FEAT_Z, TARGET
from src.evaluators.config import load_data, load_models
from src.evaluators.resampling import logo_cross_valid, logo_cross_valid_frozen
from src.data.helpers import load_perturbs

## Initalization

In [19]:
## setup
data = load_data()
models = load_models()
data_pert = load_perturbs()

## Training

In [ ]:
## feature mapping: perturbation type to feature columns
FEAT_MAP = {
    "network":    FEAT_X,
    "invariants": FEAT_X,
    "process":    FEAT_Z,
    "signature":  FEAT_Z,
    "temporal":   FEAT_Z,
}

## json key to perturbation type
KEY_TO_TYPE = {
    "network_perturbed":    "network",
    "invariants_perturbed": "invariants",
    "process_perturbed":    "process",
    "signatures_perturbed": "signature",
    "temporal_aggregated":  "temporal",
}

## worker: process a single (model, perturbation, method, intensity) combination
def _run_perturbation(model_name, model, pert_type, method, intensity, pert_df, data, feat_cols):
    lookup = pert_df.set_index("dataset")
    data_mod = data.copy()
    for col in feat_cols:
        if col in lookup.columns:
            data_mod[col] = data_mod["name"].map(lookup[col])

    ## temporal perturbation: also replace target
    if pert_type == "temporal" and TARGET in lookup.columns:
        data_mod[TARGET] = data_mod["name"].map(lookup[TARGET])

    ## drop rows without perturbation data
    required = list(feat_cols)
    if pert_type == "temporal":
        required = required + [TARGET]
    data_mod = data_mod.dropna(subset = required).reset_index(drop = True)
    if len(data_mod) < 2:
        return None

    key = (model_name, pert_type, method, intensity)

    ## retrain robustness: refit under perturbation
    frontier_rt, _ = logo_cross_valid(
        data = data_mod,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        group = "domain"
    )
    frontier_rt["model"] = model_name
    frontier_rt["perturbation"] = pert_type
    frontier_rt["method"] = method
    frontier_rt["intensity"] = intensity

    ## fixed manifold: train on clean, evaluate on perturbed
    frontier_fr, _, meta = logo_cross_valid_frozen(
        data_train = data,
        data_test = data_mod,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        group = "domain"
    )
    frontier_fr["model"] = model_name
    frontier_fr["perturbation"] = pert_type
    frontier_fr["method"] = method
    frontier_fr["intensity"] = intensity

    return {"key": key, "retrain": frontier_rt, "frozen": frontier_fr, "meta": meta}

## leave-one-group-out cross validation
results_retrain = dict()
results_frozen = dict()
frozen_metadata = dict()

## baselines (one per model)
for model_name, model in models.items():
    frontier_base, _ = logo_cross_valid(
        data = data,
        feat_x = FEAT_X,
        feat_z = FEAT_Z,
        estimator_c = model.estimator_c,
        estimator_r = model.estimator_r,
        target = TARGET,
        group = "domain"
    )
    frontier_base["model"] = model_name
    results_retrain[(model_name, "baseline", None, None)] = frontier_base
    results_frozen[(model_name, "baseline", None, None)] = frontier_base

## build job list
jobs = []
for model_name, model in models.items():
    for json_key, methods in data_pert.items():
        pert_type = KEY_TO_TYPE.get(json_key)
        if pert_type not in FEAT_MAP:
            continue
        feat_cols = FEAT_MAP[pert_type]
        for method, intensities in methods.items():
            for intensity, pert_df in intensities.items():
                jobs.append((model_name, model, pert_type, method, intensity, pert_df, data, feat_cols))

## parallel execution (n_jobs=-1 uses all cores)
print(f"Running {len(jobs)} perturbation jobs in parallel...")
outputs = Parallel(n_jobs = -1, verbose = 10)(
    delayed(_run_perturbation)(*args) for args in jobs
)

## collect results
n_ok = 0
for result in outputs:
    if result is None:
        continue
    key = result["key"]
    results_retrain[key] = result["retrain"]
    results_frozen[key] = result["frozen"]
    frozen_metadata[key] = result["meta"]
    n_ok += 1

print(f"Done. {n_ok} / {len(jobs)} jobs succeeded.")

Running 819 perturbation jobs in parallel...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:    3.0s
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    3.1s
[Parallel(n_jobs=-1)]: Done  29 tasks      | elapsed:    3.2s
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:    3.3s
[Parallel(n_jobs=-1)]: Done  53 tasks      | elapsed:    3.4s
[Parallel(n_jobs=-1)]: Done  66 tasks      | elapsed:    3.5s
[Parallel(n_jobs=-1)]: Done  81 tasks      | elapsed:    3.6s
[Parallel(n_jobs=-1)]: Done  96 tasks      | elapsed:    3.8s
[Parallel(n_jobs=-1)]: Done 113 tasks      | elapsed:    3.9s
[Parallel(n_jobs=-1)]: Done 130 tasks      | elapsed:    4.1s
[Parallel(n_jobs=-1)]: Done 149 tasks      | elapsed:    4.2s
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:    4.4s
[Parallel(n_jobs=-1)]: Done 189 tasks      | elapsed:    5.3s
[Parallel(n_jobs=-1)]: Done 210 tasks      | elapsed:    6.3s
[Parallel(n_jobs=-1)]: Done 233 tasks      | elapsed:  

Done. 819 / 819 jobs succeeded.


[Parallel(n_jobs=-1)]: Done 819 out of 819 | elapsed: 12.7min finished


## Post-Processing

In [48]:
## frontier metrics
FEAT_FRONTIER = ["vr", "mv", "ms", "ea", "ei"]

## aggregate frontier metrics across groups per perturbation setting
def _aggregate_frontier(results_dict: dict, track: str) -> pd.DataFrame:
    rows = []
    for (model_name, pert_type, method, intensity), frontier in results_dict.items():
        vals = frontier[FEAT_FRONTIER]
        row = {
            "track": track,
            "model": model_name,
            "perturbation": pert_type,
            "method": method,
            "intensity": intensity,
        }
        for col in FEAT_FRONTIER:
            row[col] = vals[col].mean()
        rows.append(row)
    return pd.DataFrame(rows)

## build results tables for both tracks
results_rt = _aggregate_frontier(results_dict = results_retrain, track = "retrain")
results_fr = _aggregate_frontier(results_dict = results_frozen, track = "frozen")
results_data = pd.concat(objs = [results_rt, results_fr], ignore_index = True)

## Directed Deltas and Recovery Ratio

In [52]:
## baseline lookup
baseline_lookup = (
    results_data.query("perturbation == 'baseline'")
    .drop_duplicates(subset = ["model"])
    .set_index("model")[FEAT_FRONTIER]
)

## directed deltas (positive = degradation)
FEAT_DELTA = [f"d_{c}" for c in FEAT_FRONTIER]
perturbed_all = results_data.query("perturbation != 'baseline'").copy()
for col in FEAT_FRONTIER:
    base = perturbed_all["model"].map(baseline_lookup[col])
    perturbed_all[f"d_{col}"] = (base - perturbed_all[col]) if col == "ei" else (perturbed_all[col] - base)

## recovery ratio: rho = 1 - d_retrain / d_frozen (only when d_frozen > eps)
idx = ["model", "perturbation", "method", "intensity"]
d_frozen = perturbed_all.query("track == 'frozen'").set_index(idx)[FEAT_DELTA]
d_retrain = perturbed_all.query("track == 'retrain'").set_index(idx)[FEAT_DELTA]
common = d_frozen.index.intersection(d_retrain.index)
d_frozen, d_retrain = d_frozen.loc[common], d_retrain.loc[common]

eps = pd.DataFrame({
    f"d_{m}": common.get_level_values("model").map(
        lambda mod, metric = m: max(0.01 * abs(baseline_lookup.loc[mod, metric]), 1e-6)
    ) for m in FEAT_FRONTIER
}, index = common)

## mask: rho is only defined when frozen degradation is meaningfully positive
eligible = d_frozen > eps
rho = (1 - d_retrain / d_frozen).where(eligible)

recovery_df = rho.reset_index()
recovery_df.columns = [c.replace("d_", "rho_") if c.startswith("d_") else c for c in recovery_df.columns]

## Manuscript Table

In [53]:
## display settings
pd.set_option("display.float_format", lambda x: f"{x:.6f}")
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

## table 1: frozen-track perturbation deltas (stability)
frozen = perturbed_all.query("track == 'frozen'")
frozen_tbl = frozen.groupby("perturbation")[FEAT_DELTA].mean().round(4)
print("=== Table 1: Frozen-Track Perturbation Deltas (Δ_frozen) ===")
display(frozen_tbl)

## table 2: retrain-track perturbation deltas (recoverability)
retrain = perturbed_all.query("track == 'retrain'")
retrain_tbl = retrain.groupby("perturbation")[FEAT_DELTA].mean().round(4)
print("\n=== Table 2: Retrain-Track Perturbation Deltas (Δ_retrain) ===")
display(retrain_tbl)

## table 3: recovery ratios
FEAT_RHO_DISPLAY = [f"rho_{c}" for c in ["ei", "vr", "mv", "ms", "ea"]]
recovery_tbl = recovery_df.groupby("perturbation")[FEAT_RHO_DISPLAY].median().round(4)
print("\n=== Table 3: Recovery Ratios (ρ) ===")
display(recovery_tbl)

=== Table 1: Frozen-Track Perturbation Deltas (Δ_frozen) ===


,d_vr,d_mv,d_ms,d_ea,d_ei
perturbation,,,,,
invariants,-0.003100,-0.022200,0.205400,0.058200,0.002300
network,0.007100,0.126400,0.243200,0.065400,0.010500
process,-0.007200,-0.115900,1229.900000,124.961500,0.008800
signature,-0.005500,-0.030000,0.218900,0.063100,0.000300
temporal,0.086800,0.529800,76.043600,5.229700,0.066000



=== Table 2: Retrain-Track Perturbation Deltas (Δ_retrain) ===


,d_vr,d_mv,d_ms,d_ea,d_ei
perturbation,,,,,
invariants,-0.008600,0.638400,1.683300,0.291200,0.009100
network,0.005500,15.601700,21.759500,2.940500,0.030100
process,-0.006100,-0.076200,1.377300,0.207700,0.007500
signature,-0.009100,0.024700,0.261700,0.083900,0.002900
temporal,-0.014400,-0.040700,3.742700,0.183700,-0.012600



=== Table 3: Recovery Ratios (ρ) ===


,rho_ei,rho_vr,rho_mv,rho_ms,rho_ea
perturbation,,,,,
invariants,0.759900,1.437500,1.519500,-0.291300,-0.380900
network,0.897500,1.000000,1.269200,0.163300,-0.380900
process,0.711200,1.000000,1.000000,0.512900,0.148800
signature,0.418900,0.833300,0.680700,-0.136900,-0.249500
temporal,1.070300,1.055600,1.204300,0.772700,0.969000
